### Carga de datos.


XM es la distruibuidora energética mayoritaria de Colombia, ellos tienen registros de hora a hora del consumo energético del país mediante la página: https://www.simem.co/.

Para la extracción de datos usaremos su API `pydataxm`

In [2]:
!pip install pydataxm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


Cómo la API tiene limitada la descarga de datos solo mes a mes, creamos este código para poderlo descargar desde un rango de 2024-2025.

In [2]:
import pandas as pd
from pydataxm.pydatasimem import ReadSIMEM
from datetime import datetime, timedelta

def descargar_rango(dataset_id, fecha_inicio, fecha_fin):
    start = datetime.strptime(fecha_inicio, '%Y-%m-%d')
    end = datetime.strptime(fecha_fin, '%Y-%m-%d')

    dfs = []
    current = start
    while current < end:
        chunk_end = min(current + timedelta(days=29), end)
        df = ReadSIMEM(
            dataset_id,
            current.strftime('%Y-%m-%d'),
            chunk_end.strftime('%Y-%m-%d')
        ).main(filter=False)
        dfs.append(df)
        current = chunk_end + timedelta(days=1)

    return pd.concat(dfs, ignore_index=True)

# Descargar 2 año completo de demanda
df_demanda = descargar_rango('c1b851', '2024-01-01', '2025-12-31')
df_demanda.to_csv('demanda_colombia.csv', index=False)

****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Datos soporte del proceso de Demandas por Código sic agente, Mercado comercialización, Sistema transmisión, versión y horaria"
****************************************************************************************************
Inicio consulta sincronica
Creacion url: 0.0005588531494140625
Extraccion de registros: 19.841805458068848
End of data extracting process
****************************************************************************************************
****************************************************************************************************
Initializing object
The object has been initialized with the dataset: "Datos soporte del proceso de Demandas por Código sic agente, Mercado comercialización, Sistema transmisión, versión y horaria"
**************************************************************

In [30]:
df_demanda.head()

,CodigoVariable,FechaHora,CodigoDuracion,UnidadMedida,CodigoSICAgente,MercadoComercializacion,SistemaTransmision,Version,Valor
0,DdaReal,2024-01-15T00:00:00,PT1H,kWh,ENIC,ENID,STR02,TXR,34309.2695
1,DdaReal,2024-01-15T01:00:00,PT1H,kWh,ENIC,ENID,STR02,TXR,32737.8301
2,DdaReal,2024-01-15T02:00:00,PT1H,kWh,ENIC,ENID,STR02,TXR,31832.3398
3,DdaReal,2024-01-15T03:00:00,PT1H,kWh,ENIC,ENID,STR02,TXR,30708.0293
4,DdaReal,2024-01-15T04:00:00,PT1H,kWh,ENIC,ENID,STR02,TXR,29963.8203


El dataset es un conjunto de datos bastante extenso

In [60]:
df_demanda.shape

(42491832, 9)

Y tenemos 9 columnas, con los siguientes datos:

In [62]:
df_demanda.info()

<class 'pandas.DataFrame'>
RangeIndex: 42491832 entries, 0 to 42491831
Data columns (total 9 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   CodigoVariable           str    
 1   FechaHora                str    
 2   CodigoDuracion           str    
 3   UnidadMedida             str    
 4   CodigoSICAgente          str    
 5   MercadoComercializacion  str    
 6   SistemaTransmision       str    
 7   Version                  str    
 8   Valor                    float64
dtypes: float64(1), str(8)
memory usage: 4.8 GB


Donde tenemos los siguientes valores únicos:

In [64]:
df_demanda.describe(include='all').loc['unique', :]

CodigoVariable                 1
FechaHora                  17544
CodigoDuracion                 1
UnidadMedida                   1
CodigoSICAgente               69
MercadoComercializacion       28
SistemaTransmision             2
Version                       10
Valor                        NaN
Name: unique, dtype: object